In [ ]:
import os, math, time, gc
import numpy as np, pandas as pd, torch, timm, torch.nn.functional as F
from scipy.stats import rankdata
from tqdm import tqdm

dev = "cuda"
OUT = r"G:\runs"; VW = os.path.join(OUT, "vinbig")
SAVE = os.path.join(OUT, "gdro_7ep_3seeds"); os.makedirs(SAVE, exist_ok=True)

EPOCHS, SEEDS, BS = 7, [0, 1, 2], 64
LR, WD, ETA = 1e-4, 1e-4, 0.01          # ETA = Group DRO step size
NORMALIZE = True                          # divide each site's loss by its constant-predictor loss (prevents collapse)
LABELS6 = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
FIND5   = ["Cardiomegaly", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
L = 6

In [ ]:
# A = NIH
nih = pd.read_csv(r"G:\nih_split.csv")
te_i = np.where(nih.official.values == "test")[0]
tr_i = np.where(nih.split_rev.values == "train")[0]
va_i = np.where(nih.split_rev.values == "val")[0]
NIH_A = np.memmap(r"G:\nih_224.u8", dtype=np.uint8, mode="r", shape=(len(nih), 224, 224))
Yn = nih[LABELS6].values.astype(int)

# B = CheXpert
cx  = pd.read_csv(r"G:\chexpert_final_clean.csv")
cxt = cx[cx.split == "test"].reset_index(drop=True)
cxm = cx[cx.split.isin(["train", "val"])].reset_index(drop=True)
cx_tr = np.where(cxm.split.values == "train")[0]; cx_va = np.where(cxm.split.values == "val")[0]
CX_A = np.memmap(r"G:\cx_trainval_224.u8", dtype=np.uint8, mode="r", shape=(len(cxm), 224, 224))
CX_T = np.load(r"G:\cx_test_224.npy")
Ycm, Yc = cxm[LABELS6].values.astype(int), cxt[LABELS6].values.astype(int)

# C = VinBigData (no Edema label -> masked)
vb = pd.read_csv(os.path.join(VW, "vinbig_image_labels.csv")); NVB = len(vb)
VB_A = np.memmap(os.path.join(VW, "vb_train_224.u8"), dtype=np.uint8, mode="r", shape=(NVB, 224, 224))
Yvb = np.zeros((NVB, L), int)
for f in FIND5: Yvb[:, LABELS6.index(f)] = vb[f].values
VALID_C = np.array([l != "Edema" for l in LABELS6])

sp = os.path.join(OUT, "fair_ens", "vinbig_split.csv")     # same split as your other runs
if os.path.exists(sp):
    s_ = pd.read_csv(sp); vb_va = np.where(s_.split.values == "val")[0]; vb_tr = np.where(s_.split.values == "train")[0]
else:
    perm = np.random.default_rng(0).permutation(NVB); vb_va = np.sort(perm[:NVB // 5]); vb_tr = np.sort(perm[NVB // 5:])

ALL = np.ones(L, bool)
SRC = {"A": dict(arr=NIH_A, Y=Yn,  tr=tr_i,  va=va_i,  valid=ALL),
       "B": dict(arr=CX_A,  Y=Ycm, tr=cx_tr, va=cx_va, valid=ALL),
       "C": dict(arr=VB_A,  Y=Yvb, tr=vb_tr, va=vb_va, valid=VALID_C)}
TEST = {"A": (NIH_A, te_i,                Yn[te_i], ALL),     # test sets (only used when the site was NOT in training)
        "B": (CX_T,  np.arange(len(CX_T)), Yc,       ALL)}

In [ ]:
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def mauc(Y, P, valid):                       # mean AUROC over valid labels
    return float(np.nanmean([fauc(Y[:, j], P[:, j]) if valid[j] and 5 <= Y[:, j].sum() <= len(Y) - 5 else np.nan for j in range(L)]))

def per_label_auc(Y, P, valid):
    return {LABELS6[j]: (fauc(Y[:, j], P[:, j]) if valid[j] and 5 <= Y[:, j].sum() <= len(Y) - 5 else np.nan) for j in range(L)}

MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def gpu_batch(A, idx, flip=False):
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1)
    if flip:
        f = torch.rand(len(idx), device=dev) < 0.5
        x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD

@torch.inference_mode()
def predict(m, A, idx, bs=128):
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        with torch.autocast("cuda"): o = m(gpu_batch(A, idx[i:i + bs]))
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()

def val_scores(m, srcs):                     # SOURCE validation only, never the target site
    return {k: mauc(SRC[k]["Y"][SRC[k]["va"]], predict(m, SRC[k]["arr"], SRC[k]["va"]), SRC[k]["valid"]) for k in srcs}

In [ ]:
def batches_bal(srcs, seed, ep):
    rng = np.random.default_rng(seed * 100 + ep)
    n_steps = math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS); half = BS // len(srcs)
    perms = {k: rng.permutation(SRC[k]["tr"]) for k in srcs}
    for s in range(n_steps):
        yield {k: np.sort(perms[k][(np.arange(half) + s * half) % len(perms[k])]) for k in srcs}   # smaller site is cycled

In [ ]:
def base_losses(srcs):
    """Loss of a constant predictor at each site's prevalence (used to normalize hardness)."""
    out = []
    for k in srcs:
        Y = SRC[k]["Y"][SRC[k]["tr"]][:, SRC[k]["valid"]]; p = np.clip(Y.mean(0), 1e-4, 1 - 1e-4)
        out.append(float(np.mean(-(p * np.log(p) + (1 - p) * np.log(1 - p)))))
    return torch.tensor(out, device=dev)

def train_gdro(ts, seed=SEED):
    srcs = ts.split("+"); tag = ts.replace("+", "p")
    BEST = os.path.join(SAVE, f"densenet121_{tag}_gdro_e{EPOCHS}_seed{seed}.pt")
    torch.manual_seed(seed); np.random.seed(seed)

    m = timm.create_model("densenet121", pretrained=True, num_classes=L).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    spe = math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * spe)
    scaler = torch.amp.GradScaler("cuda")
    q = torch.ones(len(srcs), device=dev) / len(srcs)
    base = base_losses(srcs) if NORMALIZE else torch.ones(len(srcs), device=dev)
    best, hist = 0.0, []

    for ep in range(EPOCHS):
        m.train(); t0 = time.time(); tl = 0.0
        for batch in tqdm(batches_bal(srcs, seed, ep), total=spe, leave=False, desc=f"{ts} ep {ep + 1}/{EPOCHS}"):
            xs, ys, ms, ds = [], [], [], []
            for i, k in enumerate(srcs):
                idx = batch[k]
                xs.append(gpu_batch(SRC[k]["arr"], idx, flip=True)); ys.append(SRC[k]["Y"][idx])
                ms.append(np.tile(SRC[k]["valid"], (len(idx), 1))); ds.append(np.full(len(idx), i))
            x  = torch.cat(xs)
            y  = torch.from_numpy(np.concatenate(ys)).to(dev).float()
            mk = torch.from_numpy(np.concatenate(ms)).to(dev).float()
            d  = torch.from_numpy(np.concatenate(ds)).to(dev)

            with torch.autocast("cuda"): z = m(x)
            ls = (F.binary_cross_entropy_with_logits(z.float(), y, reduction="none") * mk).sum(1) / mk.sum(1).clamp(min=1)

            # ---- Group DRO ----
            gl = torch.stack([ls[d == i].mean() for i in range(len(srcs))]) / base   # per-site loss
            q = q * torch.exp(ETA * gl.detach()); q = q / q.sum()                    # exponentiated-gradient update
            loss = (q * gl).sum()                                                    # worst-site-weighted loss

            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sch.step(); tl += loss.item()

        vs = val_scores(m, srcs); vm = float(np.mean(list(vs.values())))
        hist.append(dict(epoch=ep + 1, loss=tl / spe, val_mean=vm, **{f"val_{k}": v for k, v in vs.items()}, **{f"q_{k}": float(q[i]) for i, k in enumerate(srcs)}))
        print(f"{ts} ep {ep + 1}/{EPOCHS} | loss {tl / spe:.4f} | source val AUROC {vs} | q={q.cpu().numpy().round(3)} | {(time.time() - t0) / 60:.1f} min")
        if vm > best: best = vm; torch.save(m.state_dict(), BEST)       # select on SOURCE validation only

    m.load_state_dict(torch.load(BEST, map_location=dev, weights_only=True))
    return m, pd.DataFrame(hist)

In [ ]:
def test_on(m, T):
    arr, idx, Y, valid = TEST[T]
    P = predict(m, arr, idx)
    return mauc(Y, P, valid), per_label_auc(Y, P, valid)

In [ ]:
import json
rows, hists = [], []
for ts, T in [("A+C", "B"), ("B+C", "A")]:            # A+C -> test B, B+C -> test A
    for seed in SEEDS:
        tag = ts.replace("+", "p"); rf = os.path.join(SAVE, f"result_{tag}_seed{seed}.json")
        if os.path.exists(rf):
            r = json.load(open(rf)); print("skip (finished):", ts, "seed", seed)
        else:
            print(f"\n===== Train {ts} | Test {T} | seed {seed} | {EPOCHS} epochs =====")
            m, hist = train_gdro(ts, seed)
            auc, per = test_on(m, T)
            hist.to_csv(os.path.join(SAVE, f"hist_{tag}_seed{seed}.csv"), index=False)
            r = dict(train=ts, test=T, seed=seed, mean_AUROC=auc, **per, **{f"q_{k}": float(hist[f"q_{k}"].iloc[-1]) for k in ts.split("+")})
            json.dump(r, open(rf, "w"))
            print(f"{ts} -> {T} seed {seed}: mean AUROC = {auc:.4f}")
            del m; gc.collect(); torch.cuda.empty_cache()
        rows.append(r)

In [ ]:
R = pd.DataFrame(rows); R.to_csv(os.path.join(SAVE, "gdro_densenet121_per_seed.csv"), index=False)
display(R.round(4))

cols = ["mean_AUROC"] + [l for l in LABELS6]
agg = R.groupby(["train", "test"])[cols].agg(["mean", "std"]).round(4)
display(agg)
agg.to_csv(os.path.join(SAVE, "gdro_densenet121_summary.csv"))

for (ts, T), g in R.groupby(["train", "test"]):
    print(f"{ts} -> {T}: mean AUROC = {g.mean_AUROC.mean():.4f} ± {g.mean_AUROC.std():.4f} (3 seeds)")

# Group DRO weights at the end of training: check they did not collapse to [1, 0]
print(R.filter(regex="^(train|seed|q_)").round(3).to_string(index=False))

In [ ]:
def save_preds(m, ts, T, seed):
    srcs = ts.split("+"); tag = ts.replace("+", "p")
    z = {}
    for k in srcs:                                              # source validation predictions (for thresholds and calibration)
        z[f"{k}_val_Y"] = SRC[k]["Y"][SRC[k]["va"]]
        z[f"{k}_val_P"] = predict(m, SRC[k]["arr"], SRC[k]["va"])
    arr, idx, Y, valid = TEST[T]                                # unseen-site test predictions
    z["test_Y"], z["test_P"], z["test_valid"] = Y, predict(m, arr, idx), valid
    path = os.path.join(SAVE, f"densenet121_{tag}_gdro_e{EPOCHS}_seed{seed}_preds.npz")
    np.savez_compressed(path, **z)
    return path

In [ ]:
import json
rows = []
for ts, T in [("A+C", "B"), ("B+C", "A")]:            # A+C -> test B, B+C -> test A
    for seed in SEEDS:
        tag = ts.replace("+", "p"); rf = os.path.join(SAVE, f"result_{tag}_seed{seed}.json")
        if os.path.exists(rf):
            r = json.load(open(rf)); print("skip (finished):", ts, "seed", seed)
        else:
            print(f"\n===== Train {ts} | Test {T} | seed {seed} | {EPOCHS} epochs =====")
            m, hist = train_gdro(ts, seed)                              # best checkpoint saved inside as .pt
            auc, per = test_on(m, T)
            hist.to_csv(os.path.join(SAVE, f"hist_{tag}_seed{seed}.csv"), index=False)   # per-epoch loss, val AUROC, q
            p = save_preds(m, ts, T, seed)                              # predictions .npz
            r = dict(train=ts, test=T, seed=seed, mean_AUROC=auc, **per,
                     **{f"q_{k}": float(hist[f"q_{k}"].iloc[-1]) for k in ts.split("+")})
            json.dump(r, open(rf, "w"))                                 # per-run result (also makes reruns resumable)
            print(f"{ts} -> {T} seed {seed}: mean AUROC = {auc:.4f} | saved {os.path.basename(p)}")
            del m; gc.collect(); torch.cuda.empty_cache()
        rows.append(r)

In [ ]:
import shutil
zpath = shutil.make_archive(os.path.join(OUT, "gdro_densenet121_7ep_3seeds_all"), "zip", SAVE)
print("zipped to:", zpath)
print(pd.Series(sorted(os.listdir(SAVE))).to_string(index=False))